# Step 6: Full RAG Pipeline Implementation

This notebook completes Step 6 by integrating the validated Step 5 retriever with a Gemini generation model on Vertex AI.

Scope:
- Reuse the processed household energy chunks and embeddings.
- Connect to the deployed Vertex AI Vector Search index when available.
- Retrieve top-K chunks for a user query.
- Build a grounded prompt from the retrieved context.
- Generate an answer with Gemini on Vertex AI.
- Validate an end-to-end RAG flow: query -> retrieve -> augment -> generate.


## Cell Guide: Install Dependencies

This cell installs the Python packages required for the retriever, Vertex AI connection, and Gemini-based generation.


In [ ]:
!pip install -q google-cloud-aiplatform google-cloud-storage google-genai python-dotenv sentence-transformers pandas numpy


## Cell Guide: Load Libraries and Environment

This cell imports the required libraries, loads environment variables from `.env`, initializes Vertex AI, and creates the Gemini client.


In [1]:
import os
import re
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from google.api_core.exceptions import GoogleAPICallError, ServiceUnavailable
from google.auth.exceptions import DefaultCredentialsError
from google.cloud import aiplatform
from google import genai
from sentence_transformers import SentenceTransformer

load_dotenv(find_dotenv(usecwd=True), override=False)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent

REQUIRED_ENV_VARS = [
    "PROJECT_ID",
    "REGION",
    "DEPLOYED_INDEX_ID",
    "EMBEDDING_MODEL_NAME",
]
missing = [key for key in REQUIRED_ENV_VARS if not os.getenv(key)]
if missing:
    raise ValueError(
        "Missing required environment variables: "
        + ", ".join(missing)
        + ". Update your .env before running Step 6."
    )

CONFIG = {key: os.getenv(key) for key in REQUIRED_ENV_VARS}
CONFIG["VERTEX_ENDPOINT_ID"] = os.getenv("VERTEX_ENDPOINT_ID", "")
CONFIG["VERTEX_ENDPOINT_RESOURCE_NAME"] = os.getenv("VERTEX_ENDPOINT_RESOURCE_NAME", "")
CONFIG["GENERATION_MODEL"] = os.getenv("GENERATION_MODEL", "gemini-2.5-flash")
CONFIG["RAG_TOP_K"] = int(os.getenv("RAG_TOP_K", "4"))

RUN_STATE = {}


def mark_step(step, status, detail):
    RUN_STATE[step] = {
        "status": status,
        "detail": detail,
        "timestamp": datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z"),
    }


def auth_help():
    return (
        "Google Cloud authentication is required. Run:\n"
        "gcloud auth application-default login\n"
        f"gcloud config set project {CONFIG['PROJECT_ID']}"
    )

try:
    aiplatform.init(project=CONFIG["PROJECT_ID"], location=CONFIG["REGION"])
    gemini_client = genai.Client(vertexai=True, project=CONFIG["PROJECT_ID"], location=CONFIG["REGION"])
except DefaultCredentialsError as exc:
    raise RuntimeError(auth_help()) from exc

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ID:", CONFIG["PROJECT_ID"])
print("REGION:", CONFIG["REGION"])
print("DEPLOYED_INDEX_ID:", CONFIG["DEPLOYED_INDEX_ID"])
print("GENERATION_MODEL:", CONFIG["GENERATION_MODEL"])
mark_step("environment_loaded", "passed", "Environment variables loaded and Vertex AI / Gemini clients initialized.")


d:\Intern\venv\Lib\site-packages\google\cloud\aiplatform\models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils
d:\Intern\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: D:\Intern
PROJECT_ID: rag-project-495601
REGION: asia-southeast1
DEPLOYED_INDEX_ID: rag_energy_test_v1
GENERATION_MODEL: gemini-2.5-flash


## Cell Guide: Load Retriever Artifacts

This cell loads the chunk table and embedding matrix produced in Step 5 so the RAG pipeline can reuse them for retrieval and local fallback.


In [2]:
artifact_dir = PROJECT_ROOT / "data" / "artifacts" / "vertex_vector_search"
chunk_csv_path = artifact_dir / "energy_chunks.csv"
embeddings_npy_path = artifact_dir / "energy_embeddings.npy"

if not chunk_csv_path.exists() or not embeddings_npy_path.exists():
    raise FileNotFoundError(
        "Missing Step 5 artifacts. Run Step 5 export first so energy_chunks.csv and energy_embeddings.npy exist."
    )

chunk_df = pd.read_csv(chunk_csv_path)
embeddings = np.load(embeddings_npy_path)
embedding_model = SentenceTransformer(CONFIG["EMBEDDING_MODEL_NAME"])

print("Loaded chunks:", len(chunk_df))
print("Embedding shape:", embeddings.shape)
display(chunk_df.head(3))
mark_step("artifacts_loaded", "passed", f"Loaded {len(chunk_df)} chunks and embedding matrix {embeddings.shape}.")


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7173.04it/s]


Loaded chunks: 23
Embedding shape: (23, 1024)


,chunk_id,content,char_count,source_file,source_path,page_number,section_tag
0,energy_chunk_0000,# Document: Household Energy Cost Analysis\n# ...,140,energy_for_rag.md,D:\Intern\data\outputs\energy_for_rag.md,1,assumptions
1,energy_chunk_0001,## Page 2\n\nPurpose Methodology Acquisition c...,791,energy_for_rag.md,D:\Intern\data\outputs\energy_for_rag.md,2,assumptions
2,energy_chunk_0002,"appliances/vehicle, at cost parity)\n•\nInclud...",259,energy_for_rag.md,D:\Intern\data\outputs\energy_for_rag.md,2,general


## Cell Guide: Connect to the Vertex Index Endpoint

This cell connects to the deployed Vertex AI Vector Search endpoint so the retriever can use the live index when available.


In [3]:
vertex_endpoint = None
if CONFIG["VERTEX_ENDPOINT_RESOURCE_NAME"]:
    endpoint_name = CONFIG["VERTEX_ENDPOINT_RESOURCE_NAME"]
elif CONFIG["VERTEX_ENDPOINT_ID"]:
    endpoint_name = f"projects/{CONFIG['PROJECT_ID']}/locations/{CONFIG['REGION']}/indexEndpoints/{CONFIG['VERTEX_ENDPOINT_ID']}"
else:
    endpoint_name = ""

if endpoint_name:
    vertex_endpoint = aiplatform.MatchingEngineIndexEndpoint(endpoint_name)
    deployed_indexes = getattr(vertex_endpoint.gca_resource, "deployed_indexes", [])
    print("vertex_endpoint:", endpoint_name)
    print("deployed_indexes:", [item.id for item in deployed_indexes])
    mark_step("vertex_endpoint", "passed", f"Connected to endpoint {endpoint_name}.")
else:
    print("No endpoint configured. The notebook will use local retrieval fallback.")
    mark_step("vertex_endpoint", "skipped", "No Vertex endpoint configured; local retrieval fallback will be used.")


vertex_endpoint: projects/313902583160/locations/asia-southeast1/indexEndpoints/7725529336568086528
deployed_indexes: ['rag_energy_test_v1']


## Cell Guide: Build the Step 6 Retriever

This cell defines the retrieval helpers used by the full RAG pipeline, including Vertex retrieval, local fallback, and simple hybrid scoring.


In [4]:
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "for", "with", "on", "is", "are", "be", "by",
    "what", "which", "how", "when", "where", "why", "it", "this", "that", "from", "as", "at",
}
chunk_id_to_index = {chunk_id: idx for idx, chunk_id in enumerate(chunk_df["chunk_id"].tolist())}
id_to_row = chunk_df.set_index("chunk_id").to_dict(orient="index")


def tokenize(text):
    return [token for token in re.findall(r"[a-z0-9]+", str(text).lower()) if token not in STOPWORDS]


def keyword_score(query, text):
    query_tokens = tokenize(query)
    text_tokens = tokenize(text)
    if not query_tokens or not text_tokens:
        return 0.0
    text_counts = pd.Series(text_tokens).value_counts().to_dict()
    overlap = sum(min(text_counts.get(token, 0), 1) for token in query_tokens)
    return float(overlap / max(len(set(query_tokens)), 1))


def local_dense_retrieve(query, candidate_k=8):
    query_vector = embedding_model.encode([f"query: {query}"], normalize_embeddings=True, convert_to_numpy=True)[0]
    scores = embeddings @ query_vector
    top_indices = np.argsort(-scores)[:candidate_k]
    result = chunk_df.iloc[top_indices].copy().reset_index(drop=True)
    result["dense_score"] = scores[top_indices]
    result["dense_distance"] = 1 - result["dense_score"]
    result["retrieval_backend"] = "local_fallback"
    return result


def vertex_dense_retrieve(query, candidate_k=8):
    if vertex_endpoint is None:
        return pd.DataFrame()
    try:
        query_vector = embedding_model.encode([f"query: {query}"], normalize_embeddings=True, convert_to_numpy=True)[0].astype(float).tolist()
        response = vertex_endpoint.find_neighbors(
            deployed_index_id=CONFIG["DEPLOYED_INDEX_ID"],
            queries=[query_vector],
            num_neighbors=candidate_k,
            return_full_datapoint=True,
        )
    except (ServiceUnavailable, GoogleAPICallError, Exception) as exc:
        print(f"Vertex retrieval fallback triggered: {exc}")
        return pd.DataFrame()

    if not response or not response[0]:
        return pd.DataFrame()

    rows = []
    for neighbor in response[0]:
        metadata = getattr(neighbor, "embedding_metadata", None) or {}
        local_row = id_to_row.get(neighbor.id, {})
        rows.append({
            "chunk_id": neighbor.id,
            "content": metadata.get("content") or local_row.get("content"),
            "source_file": metadata.get("source_file") or local_row.get("source_file"),
            "source_path": metadata.get("source_path") or local_row.get("source_path"),
            "page_number": metadata.get("page_number") or local_row.get("page_number"),
            "section_tag": metadata.get("section_tag") or local_row.get("section_tag"),
            "dense_distance": getattr(neighbor, "distance", None),
            "dense_score": 1 - float(getattr(neighbor, "distance", 1.0)),
            "retrieval_backend": "vertex",
        })
    return pd.DataFrame(rows)


def retrieve_context(query, top_k=4, candidate_k=8):
    dense_df = vertex_dense_retrieve(query, candidate_k=candidate_k)
    if dense_df.empty:
        dense_df = local_dense_retrieve(query, candidate_k=candidate_k)

    rerank_df = dense_df.copy()
    rerank_df["keyword_score"] = rerank_df["content"].apply(lambda text: keyword_score(query, text))
    rerank_df["hybrid_score"] = (0.75 * rerank_df["dense_score"]) + (0.25 * rerank_df["keyword_score"])
    rerank_df = rerank_df.sort_values(["hybrid_score", "dense_score"], ascending=[False, False]).head(top_k).reset_index(drop=True)
    return rerank_df

mark_step("retriever_ready", "passed", "Built Step 6 retrieval helpers with Vertex retrieval and local fallback.")
print(RUN_STATE["retriever_ready"])


{'status': 'passed', 'detail': 'Built Step 6 retrieval helpers with Vertex retrieval and local fallback.', 'timestamp': '2026-05-14T03:14:10Z'}


## Cell Guide: Build the Grounded RAG Prompt

This cell formats retrieved chunks into a grounded context block and constructs the final prompt sent to Gemini.


In [5]:
def format_context_for_prompt(retrieval_df):
    sections = []
    for i, row in retrieval_df.iterrows():
        sections.append(
            f"[Context {i + 1} | Chunk: {row['chunk_id']} | Page: {row.get('page_number', 'N/A')} | Section: {row.get('section_tag', 'general')}]\\n{row['content']}"
        )
    return "\\n\\n".join(sections)


def build_rag_prompt(query, retrieval_df):
    context_block = format_context_for_prompt(retrieval_df)
    prompt = f"""
You are a grounded RAG assistant for a household energy analysis project.
Use only the provided context to answer the question.
If the context is insufficient, say so clearly instead of inventing facts.
Cite the relevant chunk IDs in your answer when possible.

Question:
{query}

Retrieved Context:
{context_block}

Answer:
""".strip()
    return prompt

mark_step("prompt_ready", "passed", "Built prompt formatting helpers for grounded generation.")
print(RUN_STATE["prompt_ready"])


{'status': 'passed', 'detail': 'Built prompt formatting helpers for grounded generation.', 'timestamp': '2026-05-14T03:14:10Z'}


## Cell Guide: Generate the Final RAG Answer

This cell defines the Gemini generation helper and combines retrieval, prompt augmentation, and generation into one end-to-end RAG function.


In [6]:
def generate_with_gemini(prompt, model=None):
    model_name = model or CONFIG["GENERATION_MODEL"]
    response = gemini_client.models.generate_content(model=model_name, contents=prompt)
    return response.text


def ask_energy_rag(query, top_k=None, model=None):
    retrieval_df = retrieve_context(query=query, top_k=top_k or CONFIG["RAG_TOP_K"])
    prompt = build_rag_prompt(query, retrieval_df)
    answer = generate_with_gemini(prompt=prompt, model=model)
    return {
        "query": query,
        "retrieval_df": retrieval_df,
        "prompt": prompt,
        "answer": answer,
    }

mark_step("generator_ready", "passed", "Built Gemini generation helper and the full Step 6 ask_energy_rag function.")
print(RUN_STATE["generator_ready"])


{'status': 'passed', 'detail': 'Built Gemini generation helper and the full Step 6 ask_energy_rag function.', 'timestamp': '2026-05-14T03:14:10Z'}


## Cell Guide: Run the End-to-End RAG Test

This cell runs the full Step 6 pipeline on sample dataset questions and prints the retrieved evidence and generated answers.


In [7]:
sample_queries = [
    "What assumptions are used in the household energy cost analysis?",
    "Which household types are included in the analysis?",
]

rag_runs = []
for query in sample_queries:
    result = ask_energy_rag(query=query)
    rag_runs.append({
        "query": result["query"],
        "answer": result["answer"],
        "retrieval_backend": result["retrieval_df"]["retrieval_backend"].iloc[0] if not result["retrieval_df"].empty else "none",
        "top_chunk_ids": ", ".join(result["retrieval_df"]["chunk_id"].tolist()),
    })
    print("=" * 100)
    print("QUERY:", result["query"])
    print("\\nRETRIEVED CHUNKS:")
    display(result["retrieval_df"][["chunk_id", "retrieval_backend", "dense_score", "keyword_score", "hybrid_score", "content"]])
    print("\\nGENERATED ANSWER:\\n")
    print(result["answer"])

rag_results_df = pd.DataFrame(rag_runs)
display(rag_results_df)
mark_step("rag_pipeline", "passed", "Executed end-to-end RAG queries with retrieval, prompt augmentation, and Gemini generation.")
print(RUN_STATE["rag_pipeline"])


QUERY: What assumptions are used in the household energy cost analysis?
\nRETRIEVED CHUNKS:


,chunk_id,retrieval_backend,dense_score,keyword_score,hybrid_score,content
0,energy_chunk_0000,vertex,0.855461,0.833333,0.849929,# Document: Household Energy Cost Analysis\n# ...
1,energy_chunk_0006,vertex,0.840878,0.666667,0.797325,## Page 3\n\nPurpose Methodology Acquisition c...
2,energy_chunk_0011,vertex,0.842710,0.500000,0.757032,## Page 4\n\nPurpose Methodology Acquisition c...
3,energy_chunk_0004,vertex,0.822349,0.500000,0.741762,. upfront purchase costs of electrical applian...


\nGENERATED ANSWER:\n
The household energy cost analysis uses the following assumptions:

*   **General Assumptions**
    *   Upfront purchase costs of electrical appliances and EVs are excluded, based on the assumption that these purchases will be made at the end of life of existing appliances/vehicles at cost parity (energy_chunk_0004).
    *   The analysis is expressed in real 2026 dollars ($2026) (energy_chunk_0004).
    *   The analysis includes three different household types, both with and without solar PV: Dual fuel + ICE vehicle, Electrified household + ICE vehicle, and Electrified household + EV (energy_chunk_0004).

*   **Usage Profiles (Current as at July 2024)**
    *   Gas usage: 49,800 MJ per household (energy_chunk_0006).
    *   Petrol/ICE vehicles: 1 vehicle per household, 11,100 km/year for passenger vehicles with 11.1 L/100km fuel efficiency (energy_chunk_0006).
    *   Electric Vehicles (EVs): 1 vehicle per household, 11,000 km/year (energy_chunk_0006).
    *   Seg

,chunk_id,retrieval_backend,dense_score,keyword_score,hybrid_score,content
0,energy_chunk_0002,vertex,0.806498,0.5,0.729873,"appliances/vehicle, at cost parity)\n•\nInclud..."
1,energy_chunk_0004,vertex,0.798687,0.5,0.724015,. upfront purchase costs of electrical applian...
2,energy_chunk_0000,vertex,0.789218,0.5,0.716913,# Document: Household Energy Cost Analysis\n# ...
3,energy_chunk_0006,vertex,0.787410,0.5,0.715558,## Page 3\n\nPurpose Methodology Acquisition c...


\nGENERATED ANSWER:\n
The analysis includes the following three household types, both with and without solar PV:
*   Dual fuel + ICE vehicle
*   Electrified household + ICE vehicle
*   Electrified household + EV

(Context 1, Context 2)


,query,answer,retrieval_backend,top_chunk_ids
0,What assumptions are used in the household ene...,The household energy cost analysis uses the fo...,vertex,"energy_chunk_0000, energy_chunk_0006, energy_c..."
1,Which household types are included in the anal...,The analysis includes the following three hous...,vertex,"energy_chunk_0002, energy_chunk_0004, energy_c..."


{'status': 'passed', 'detail': 'Executed end-to-end RAG queries with retrieval, prompt augmentation, and Gemini generation.', 'timestamp': '2026-05-14T03:14:16Z'}


## Cell Guide: Summarize Step 6 Validation

This cell summarizes whether the full RAG pipeline is functional and ready for the next evaluation stage.


In [8]:
test_results_df = pd.DataFrame([
    {"step": step, **payload}
    for step, payload in RUN_STATE.items()
])
display(test_results_df)

summary_rows = [
    {"test_area": "Environment setup", "status": RUN_STATE.get("environment_loaded", {}).get("status", "not_run"), "evidence": RUN_STATE.get("environment_loaded", {}).get("detail", "")},
    {"test_area": "Artifacts loaded", "status": RUN_STATE.get("artifacts_loaded", {}).get("status", "not_run"), "evidence": RUN_STATE.get("artifacts_loaded", {}).get("detail", "")},
    {"test_area": "Retriever ready", "status": RUN_STATE.get("retriever_ready", {}).get("status", "not_run"), "evidence": RUN_STATE.get("retriever_ready", {}).get("detail", "")},
    {"test_area": "Prompt builder", "status": RUN_STATE.get("prompt_ready", {}).get("status", "not_run"), "evidence": RUN_STATE.get("prompt_ready", {}).get("detail", "")},
    {"test_area": "Generator ready", "status": RUN_STATE.get("generator_ready", {}).get("status", "not_run"), "evidence": RUN_STATE.get("generator_ready", {}).get("detail", "")},
    {"test_area": "End-to-end RAG", "status": RUN_STATE.get("rag_pipeline", {}).get("status", "not_run"), "evidence": RUN_STATE.get("rag_pipeline", {}).get("detail", "")},
]
step6_summary_df = pd.DataFrame(summary_rows)
display(step6_summary_df)

step6_ready = all(row["status"] == "passed" for row in summary_rows)
if step6_ready:
    print("Step 6 milestone reached: the notebook demonstrates a functional RAG system on the test dataset.")
else:
    print("Step 6 is partially complete. Review the summary table for the remaining gaps.")


,step,status,detail,timestamp
0,environment_loaded,passed,Environment variables loaded and Vertex AI / G...,2026-05-14T03:13:59Z
1,artifacts_loaded,passed,"Loaded 23 chunks and embedding matrix (23, 1024).",2026-05-14T03:14:07Z
2,vertex_endpoint,passed,Connected to endpoint projects/313902583160/lo...,2026-05-14T03:14:10Z
3,retriever_ready,passed,Built Step 6 retrieval helpers with Vertex ret...,2026-05-14T03:14:10Z
4,prompt_ready,passed,Built prompt formatting helpers for grounded g...,2026-05-14T03:14:10Z
5,generator_ready,passed,Built Gemini generation helper and the full St...,2026-05-14T03:14:10Z
6,rag_pipeline,passed,Executed end-to-end RAG queries with retrieval...,2026-05-14T03:14:16Z


,test_area,status,evidence
0,Environment setup,passed,Environment variables loaded and Vertex AI / G...
1,Artifacts loaded,passed,"Loaded 23 chunks and embedding matrix (23, 1024)."
2,Retriever ready,passed,Built Step 6 retrieval helpers with Vertex ret...
3,Prompt builder,passed,Built prompt formatting helpers for grounded g...
4,Generator ready,passed,Built Gemini generation helper and the full St...
5,End-to-end RAG,passed,Executed end-to-end RAG queries with retrieval...


Step 6 milestone reached: the notebook demonstrates a functional RAG system on the test dataset.
